Langchain + SQL

In [ ]:
from langchain_community.utilities import SQLDatabase

In [6]:
db_user="root"
db_password=""
db_host="localhost"
db_port=3306
db_name="atliq_tshirts"
db=SQLDatabase.from_uri(
    f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
,sample_rows_in_table_info=5)


print(db.table_info)



CREATE TABLE discounts (
	discount_id INTEGER NOT NULL AUTO_INCREMENT, 
	t_shirt_id INTEGER NOT NULL, 
	pct_discount DECIMAL(5, 2), 
	PRIMARY KEY (discount_id), 
	CONSTRAINT discounts_ibfk_1 FOREIGN KEY(t_shirt_id) REFERENCES t_shirts (t_shirt_id), 
	CONSTRAINT discounts_chk_1 CHECK ((`pct_discount` between 0 and 100))
)DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci ENGINE=InnoDB

/*
5 rows from discounts table:
discount_id	t_shirt_id	pct_discount
1	1	10.00
2	2	15.00
3	3	20.00
4	4	5.00
5	5	25.00
*/


CREATE TABLE t_shirts (
	t_shirt_id INTEGER NOT NULL AUTO_INCREMENT, 
	brand ENUM('Van Huesen','Levi','Nike','Adidas') NOT NULL, 
	color ENUM('Red','Blue','Black','White') NOT NULL, 
	size ENUM('XS','S','M','L','XL') NOT NULL, 
	price INTEGER, 
	stock_quantity INTEGER NOT NULL, 
	PRIMARY KEY (t_shirt_id), 
	CONSTRAINT t_shirts_chk_1 CHECK ((`price` between 10 and 50))
)DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci ENGINE=InnoDB

/*
5 rows from t_shirts table:
t_shirt_id	brand	col

In [44]:
from gpt_client import llm;
from langchain_experimental.sql import SQLDatabaseChain; 

db_chain = SQLDatabaseChain.from_llm(llm, db, verbose=True)

qns1=db_chain.run("How many tshirts do we have left for nike in extra small size and white color?")



> Entering new SQLDatabaseChain chain...
How many tshirts do we have left for nike in extra small size and white color?
SQLQuery:SQLQuery: SELECT `stock_quantity` FROM `t_shirts` WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'White';
SQLResult: [(42,)]
Answer:Question: How many tshirts do we have left for nike in extra small size and white color?
SQLQuery: SELECT `stock_quantity` FROM `t_shirts` WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'White';
> Finished chain.


In [49]:
qns1=db_chain.run("SELECT `stock_quantity` FROM `t_shirts` WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'White'")



> Entering new SQLDatabaseChain chain...
SELECT `stock_quantity` FROM `t_shirts` WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'White'
SQLQuery:SQLQuery: SELECT `stock_quantity` FROM `t_shirts` WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'White'
SQLResult: [(42,)]
Answer:Question: SELECT `stock_quantity` FROM `t_shirts` WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'White'  
SQLQuery: SELECT `stock_quantity` FROM `t_shirts` WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'White'
> Finished chain.


'Q'

## how much is the price of the inventory for all small size t-shirts

In [34]:
qns2=db_chain.run("how much is the total price for all small size t-shirts?")



> Entering new SQLDatabaseChain chain...
how much is the total price for all small size t-shirts?
SQLQuery:SQLQuery: SELECT SUM(`price`) AS total_price FROM `t_shirts` WHERE `size` = 'S';
SQLResult: [(Decimal('305'),)]
Answer:Question: how much is the total price for all small size t-shirts?  
SQLQuery: SELECT SUM(`price`) AS total_price FROM `t_shirts` WHERE `size` = 'S';
> Finished chain.


In [35]:
qns2=db_chain.run("SELECT SUM(quantity*`price`) AS total_price FROM `t_shirts` WHERE `size` = 'S'")



> Entering new SQLDatabaseChain chain...
SELECT SUM(quantity*`price`) AS total_price FROM `t_shirts` WHERE `size` = 'S'
SQLQuery:Question: SELECT SUM(quantity*`price`) AS total_price FROM `t_shirts` WHERE `size` = 'S'
SQLQuery: SELECT SUM(`stock_quantity` * `price`) AS total_price FROM `t_shirts` WHERE `size` = 'S'
SQLResult: [(Decimal('17758'),)]
Answer:Question: SELECT SUM(quantity*`price`) AS total_price FROM `t_shirts` WHERE `size` = 'S'
SQLQuery: SELECT SUM(`stock_quantity` * `price`) AS total_price FROM `t_shirts` WHERE `size` = 'S'
> Finished chain.


In [36]:
qns2

"Question: SELECT SUM(quantity*`price`) AS total_price FROM `t_shirts` WHERE `size` = 'S'\nSQLQuery: SELECT SUM(`stock_quantity` * `price`) AS total_price FROM `t_shirts` WHERE `size` = 'S'"

## If i sell all my levis tshirt today with discounts applied, how much revenue our store will generate (post discounts)

In [38]:
qns3=db_chain.run("If i sell all my levis tshirt today with discounts applied, how much revenue our store will generate (post discounts)")



> Entering new SQLDatabaseChain chain...
If i sell all my levis tshirt today with discounts applied, how much revenue our store will generate (post discounts)
SQLQuery:SQLQuery: 
```sql
SELECT 
    SUM(`price` * (1 - `pct_discount` / 100)) AS `total_revenue`
FROM 
    t_shirts 
JOIN 
    discounts ON t_shirts.t_shirt_id = discounts.t_shirt_id 
WHERE 
    t_shirts.brand = 'Levi' 
    AND t_shirts.stock_quantity > 0;
```

ProgrammingError: (pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '```sql\nSELECT \n    SUM(`price` * (1 - `pct_discount` / 100)) AS `total_revenue`\n' at line 1")
[SQL: ```sql
SELECT 
    SUM(`price` * (1 - `pct_discount` / 100)) AS `total_revenue`
FROM 
    t_shirts 
JOIN 
    discounts ON t_shirts.t_shirt_id = discounts.t_shirt_id 
WHERE 
    t_shirts.brand = 'Levi' 
    AND t_shirts.stock_quantity > 0;
```]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [39]:
sql_code = """
select sum(a.total_amount * ((100-COALESCE(discounts.pct_discount,0))/100)) as total_revenue from
(select sum(price*stock_quantity) as total_amount, t_shirt_id from t_shirts where brand = 'Levi'
group by t_shirt_id) a left join discounts on a.t_shirt_id = discounts.t_shirt_id
 """

qns3 = db_chain.run(sql_code)



> Entering new SQLDatabaseChain chain...

select sum(a.total_amount * ((100-COALESCE(discounts.pct_discount,0))/100)) as total_revenue from
(select sum(price*stock_quantity) as total_amount, t_shirt_id from t_shirts where brand = 'Levi'
group by t_shirt_id) a left join discounts on a.t_shirt_id = discounts.t_shirt_id
 
SQLQuery:Question: 
select sum(a.total_amount * ((100-COALESCE(discounts.pct_discount,0))/100)) as total_revenue from
(select sum(price*stock_quantity) as total_amount, t_shirt_id from t_shirts where brand = 'Levi'
group by t_shirt_id) a left join discounts on a.t_shirt_id = discounts.t_shirt_id

SQLQuery: 
```sql
SELECT SUM(a.total_amount * ((100 - COALESCE(discounts.pct_discount, 0)) / 100)) AS total_revenue 
FROM (
    SELECT SUM(`price` * `stock_quantity`) AS total_amount, `t_shirt_id` 
    FROM `t_shirts` 
    WHERE `brand` = 'Levi' 
    GROUP BY `t_shirt_id`
) a 
LEFT JOIN `discounts` ON a.`t_shirt_id` = `discounts`.`t_shirt_id`;
```

ProgrammingError: (pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '```sql\nSELECT SUM(a.total_amount * ((100 - COALESCE(discounts.pct_discount, 0)) ' at line 1")
[SQL: ```sql
SELECT SUM(a.total_amount * ((100 - COALESCE(discounts.pct_discount, 0)) / 100)) AS total_revenue 
FROM (
    SELECT SUM(`price` * `stock_quantity`) AS total_amount, `t_shirt_id` 
    FROM `t_shirts` 
    WHERE `brand` = 'Levi' 
    GROUP BY `t_shirt_id`
) a 
LEFT JOIN `discounts` ON a.`t_shirt_id` = `discounts`.`t_shirt_id`;
```]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [17]:
qns3=db_chain.run("select sum(price*stock_quantity) from t_shirts where size = 'S'");



> Entering new SQLDatabaseChain chain...
select sum(price*stock_quantity) from t_shirts where size = 'S'
SQLQuery:SQLQuery: SELECT SUM(`price` * `stock_quantity`) AS total_value FROM `t_shirts` WHERE `size` = 'S';
SQLResult: [(Decimal('17758'),)]
Answer:Question: select sum(price*stock_quantity) from t_shirts where size = 'S'
SQLQuery: SELECT SUM(`price` * `stock_quantity`) AS total_value FROM `t_shirts` WHERE `size` = 'S';
> Finished chain.


In [ ]:
qns3

"Question: select sum(price*stock_quantity) from t_shirts where size = 'S'\nSQLQuery: SELECT SUM(`price` * `stock_quantity`) AS total_value FROM `t_shirts` WHERE `size` = 'S';"

In [41]:
qns4=db_chain.run("If we have to sell all the Levi's tshirts  how much revenue will we generate?");



> Entering new SQLDatabaseChain chain...
If we have to sell all the Levi's tshirts  how much revenue will we generate?
SQLQuery:SQLQuery: 
```sql
SELECT SUM(`price`) AS total_revenue 
FROM `t_shirts` 
WHERE `brand` = 'Levi' AND `stock_quantity` > 0;
```

ProgrammingError: (pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '```sql\nSELECT SUM(`price`) AS total_revenue \nFROM `t_shirts` \nWHERE `brand` = 'L' at line 1")
[SQL: ```sql
SELECT SUM(`price`) AS total_revenue 
FROM `t_shirts` 
WHERE `brand` = 'Levi' AND `stock_quantity` > 0;
```]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [27]:
qns10=db_chain.run("SELECT SUM(`price` * (1 - `pct_discount` / 100)) AS `revenue` FROM `t_shirts` JOIN `discounts` ON `t_shirts`.`t_shirt_id` = `discounts`.`t_shirt_id` WHERE `t_shirts`.`brand` = 'Levi' AND `t_shirts`.`stock_quantity` > 0;")



> Entering new SQLDatabaseChain chain...
SELECT SUM(`price` * (1 - `pct_discount` / 100)) AS `revenue` FROM `t_shirts` JOIN `discounts` ON `t_shirts`.`t_shirt_id` = `discounts`.`t_shirt_id` WHERE `t_shirts`.`brand` = 'Levi' AND `t_shirts`.`stock_quantity` > 0;
SQLQuery:SQLQuery: SELECT SUM(`price` * (1 - `pct_discount` / 100)) AS `revenue` FROM `t_shirts` JOIN `discounts` ON `t_shirts`.`t_shirt_id` = `discounts`.`t_shirt_id` WHERE `t_shirts`.`brand` = 'Levi' AND `t_shirts`.`stock_quantity` > 0;
SQLResult: [(None,)]
Answer:Question: SELECT SUM(`price` * (1 - `pct_discount` / 100)) AS `revenue` FROM `t_shirts` JOIN `discounts` ON `t_shirts`.`t_shirt_id` = `discounts`.`t_shirt_id` WHERE `t_shirts`.`brand` = 'Levi' AND `t_shirts`.`stock_quantity` > 0;
SQLQuery: SELECT SUM(`price` * (1 - `pct_discount` / 100)) AS `revenue` FROM `t_shirts` JOIN `discounts` ON `t_shirts`.`t_shirt_id` = `discounts`.`t_shirt_id` WHERE `t_shirts`.`brand` = 'Levi' AND `t_shirts`.`stock_quantity` > 0;
> Finished

How much revenue will i make if i sell all levis tshirt

In [ ]:
qns4=db_chain.run("If we have to sell all the Levi's tshirts, what is the total revenue?");




> Entering new SQLDatabaseChain chain...
If we have to sell all the Levi's tshirts, what is the total revenue?
SQLQuery:SQLQuery: 
```sql
SELECT SUM(`price`) AS total_revenue 
FROM `t_shirts` 
WHERE `brand` = 'Levi' AND `stock_quantity` > 0;
```

ProgrammingError: (pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '```sql\nSELECT SUM(`price`) AS total_revenue \nFROM `t_shirts` \nWHERE `brand` = 'L' at line 1")
[SQL: ```sql
SELECT SUM(`price`) AS total_revenue 
FROM `t_shirts` 
WHERE `brand` = 'Levi' AND `stock_quantity` > 0;
```]
(Background on this error at: https://sqlalche.me/e/20/f405)

How many white color levis tshirts do we have?

In [ ]:
qns5=db_chain.run("How many total white color levis tshirts we have available?");




> Entering new SQLDatabaseChain chain...
How many total white color levis tshirts we have available? (provide just the query and run)
SQLQuery:SQLQuery: 
```sql
SELECT SUM(`stock_quantity`) AS total_stock 
FROM `t_shirts` 
WHERE `color` = 'White' AND `brand` = 'Levi';
```

ProgrammingError: (pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '```sql\nSELECT SUM(`stock_quantity`) AS total_stock \nFROM `t_shirts` \nWHERE `colo' at line 1")
[SQL: ```sql
SELECT SUM(`stock_quantity`) AS total_stock 
FROM `t_shirts` 
WHERE `color` = 'White' AND `brand` = 'Levi';
```]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [29]:
qns5=db_chain.run("SELECT SUM(`stock_quantity`) AS total_white_levis FROM `t_shirts` WHERE `color` = 'White' AND `brand` = 'Levi';")



> Entering new SQLDatabaseChain chain...
SELECT SUM(`stock_quantity`) AS total_white_levis FROM `t_shirts` WHERE `color` = 'White' AND `brand` = 'Levi';
SQLQuery:SQLQuery: SELECT SUM(`stock_quantity`) AS total_white_levis FROM `t_shirts` WHERE `color` = 'White' AND `brand` = 'Levi';
SQLResult: [(Decimal('138'),)]
Answer:Question: SELECT SUM(`stock_quantity`) AS total_white_levis FROM `t_shirts` WHERE `color` = 'White' AND `brand` = 'Levi';
SQLQuery: SELECT SUM(`stock_quantity`) AS total_white_levis FROM `t_shirts` WHERE `color` = 'White' AND `brand` = 'Levi';
> Finished chain.


# Few short Learning


In [ ]:
few_shots = [
    {'Question' : "How many t-shirts do we have left for Nike in XS size and white color?",
     'SQLQuery' : "SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Nike' AND color = 'White' AND size = 'XS'",
     'SQLResult': "Result of the SQL query",
     'Answer' : qns1},
    {'Question': "How much is the total price of the inventory for all S-size t-shirts?",
     'SQLQuery':"SELECT SUM(price*stock_quantity) FROM t_shirts WHERE size = 'S'",
     'SQLResult': "Result of the SQL query",
     'Answer': qns2},
    {'Question': "If we have to sell all the Levi’s T-shirts today with discounts applied. How much revenue  our store will generate (post discounts)?" ,
     'SQLQuery' : """SELECT sum(a.total_amount * ((100-COALESCE(discounts.pct_discount,0))/100)) as total_revenue from
(select sum(price*stock_quantity) as total_amount, t_shirt_id from t_shirts where brand = 'Levi'
group by t_shirt_id) a left join discounts on a.t_shirt_id = discounts.t_shirt_id
 """,
     'SQLResult': "Result of the SQL query",
     'Answer': qns3} ,
     {'Question' : "If we have to sell all the Levi’s T-shirts today. How much revenue our store will generate without discount?" ,
      'SQLQuery': "SELECT SUM(price * stock_quantity) FROM t_shirts WHERE brand = 'Levi'",
      'SQLResult': "Result of the SQL query",
      'Answer' : qns4},
    {'Question': "How many white color Levi's shirt I have?",
     'SQLQuery' : "SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Levi' AND color = 'White'",
     'SQLResult': "Result of the SQL query",
     'Answer' : qns5
     }
]


Now adding hugging face to generate vectors

In [52]:
from langchain.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)


e=embeddings.embed_query("How many white color Levi's shirt I have?")

/var/folders/03/jb_nwrfd38g1fhb0769lf5dm0000gn/T/ipykernel_71941/1045530641.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [53]:
e[:5]

[0.003610372543334961,
 0.07093445211648941,
 -0.002751024207100272,
 0.0009241142543032765,
 0.05407032370567322]

In [ ]:
How many white color Levi's shirt I have?" "SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Levi' AND color = 'White'" "Result of the SQL query" 138

In [58]:
to_vectorize=[" ".join(map(str, example.values())) for example in few_shots]


In [59]:
from langchain.vectorstores import Chroma

vector_store = Chroma.from_texts(
    to_vectorize,
    embedding=embeddings,
    metadatas=few_shots
)



In [61]:
from langchain.prompts import SemanticSimilarityExampleSelector

example_selector = SemanticSimilarityExampleSelector(
    vectorstore=vector_store,
    k=2
)
example_selector

SemanticSimilarityExampleSelector(vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x14bfd7490>, k=2, example_keys=None, input_keys=None, vectorstore_kwargs=None)

In [64]:
example_selector.select_examples({"Question":"If we have to sell all the Levi’s T-shirts today. How much revenue our store will generate?"})

[{'SQLResult': "[(Decimal('42'),)]",
  'Question': 'How many tshirts do we have left for Nike in extra small size and white color?',
  'SQLQuery': "SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `color` = 'White' AND `brand` = 'Nike' AND `size` = 'XS';",
  'Answer': '42'}]

In [66]:
from langchain.chains.sql_database.prompt import PROMPT_SUFFIX, _mysql_prompt

In [67]:
print(_mysql_prompt)

You are a MySQL expert. Given an input question, first create a syntactically correct MySQL query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per MySQL. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in backticks (`) to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use CURDATE() function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result of

In [68]:
print(PROMPT_SUFFIX)

Only use the following tables:
{table_info}

Question: {input}


In [71]:
from langchain.prompts.prompt import PromptTemplate
from langchain.prompts import FewShotPromptTemplate

example_prompt = PromptTemplate(
    input_variables=["Question", "SQLQuery", "SQLResult","Answer",],
    template="\nQuestion: {Question}\nSQLQuery: {SQLQuery}\nSQLResult: {SQLResult}\nAnswer: {Answer}",
)


In [70]:
print(_mysql_prompt)


You are a MySQL expert. Given an input question, first create a syntactically correct MySQL query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per MySQL. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in backticks (`) to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use CURDATE() function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result of

In [72]:
few_shot_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix=_mysql_prompt,
    suffix=PROMPT_SUFFIX,
    input_variables=["input", "table_info", "top_k"], #These variables are used in the prefix and suffix
)


In [73]:
new_chain=SQLDatabaseChain.from_llm(
    llm,
    db,
    prompt=few_shot_prompt,
    verbose=True
)




In [74]:
new_chain("How many white color Levi's shirt I have?")



> Entering new SQLDatabaseChain chain...
How many white color Levi's shirt I have?
SQLQuery:SQLQuery: SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `color` = 'White' AND `brand` = 'Levi';
SQLResult: [(Decimal('138'),)]
Answer:Question: How many white color Levi's shirt I have?
SQLQuery: SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `color` = 'White' AND `brand` = 'Levi';
> Finished chain.


{'query': "How many white color Levi's shirt I have?",
 'result': "Question: How many white color Levi's shirt I have?\nSQLQuery: SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `color` = 'White' AND `brand` = 'Levi';"}

In [75]:
new_chain("If we have to sell all the Levi's tshirts  how much revenue will we generate?");



> Entering new SQLDatabaseChain chain...
If we have to sell all the Levi's tshirts  how much revenue will we generate?
SQLQuery:SQLQuery: SELECT SUM(`price` * `stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Levi';
SQLResult: [(Decimal('18532'),)]
Answer:Question: If we have to sell all the Levi's tshirts how much revenue will we generate?
SQLQuery: SELECT SUM(`price` * `stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Levi';
> Finished chain.
